# 28. In-season learning, level 1: updating projections with this season's games

**Ryan's idea:** treat each simulated season of a player as a **guess** at his true level this season, and treat real games as **evidence**.

- **Guesses that fit the evidence** get more weight.
- **The rest of the season** is simulated from the reweighted guesses.

That's Bayesian updating by importance weighting (a particle filter). It's done **one player at a time**, because each player's notebook 17 draw is independent: no single simulated league has to match every real player.

**Two versions of a guess** (code: `src/inseason.py`):

- **whole:** a guess is a whole-season outcome (talent change + that season's luck), carried into the rest of the season as is.
- **split:** a guess is split with notebook 17's luck fit into a **talent part** and **luck**. Evidence updates the talent part, and the rest of the season gets **fresh luck**. Luck in games already played shouldn't repeat.

**The rule:** an update is adopted only if a backtest on past seasons shows it predicts the rest of a season better than **both** the preseason projection and the current pace (or a blend with pace does), on the **test** seasons.

**Said plainly:**

- **Skater WAR so far** can't be backtested: past partial-season WAR doesn't exist. So it's left out of the default and shown as a separate version.
- **Goalie evidence** can be tested only on 2021-22 to 2025-26, the only seasons with shot-level goalie data.

## 1. Setup

In [1]:
import sys
sys.path.append("../src")

import json
import time
import zipfile
import numpy as np
import pandas as pd
from pathlib import Path

from forecast import history_table, project, backtest, aging_curve, group_average, current_projections, add_age, GOALS_PER_WIN
from edge_data import build_bio_table, schedule_games, nhl_code, boxscore_lines, season_schedule
from nhle import load_profiles
from development import (training_nhle, best_nhle, skater_features, fit_prior, apply_prior, candidates, fit_rows,
                         experience, TARGETS, AGING, SK_WEIGHTS, SK_REG, G_WEIGHTS, G_REG, PRIOR_INPUTS)
from war_model import run_season
from goalie_war import load_goalies
import lineups as L
import simulate as S
import scoring as SC
import season as SE
import inseason as IS

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)

DATA = Path("../data")
BUNDLE_DIR = DATA / "simulator_2026-27"
TRADES_PATH = DATA / "trades_2026-27.csv"
LOG_DIR = DATA / "forecast_log"

def label(s):
    return f"{s}-{str(s + 1)[-2:]}"

TRAIN_END = 2017
TRAIN = [t for t in TARGETS if t <= TRAIN_END]
TEST = [t for t in TARGETS if t > TRAIN_END]
BT_TRAIN = [2013, 2014, 2015, 2016, 2017]            # backtest training seasons (boxscores downloaded)
BT_TEST = [2018, 2021, 2022, 2023, 2024, 2025]       # backtest test seasons
BT = BT_TRAIN + BT_TEST
GOALIE_SEASONS = [2021, 2022, 2023, 2024, 2025]      # shot-level goalie data
CUTS = [10, 20, 40]                                  # update after the first team reaches 10, 20, 40 games
CHOSEN = {"F": (0.75, "R1"), "D": (0.75, "R1"), "G": (0.5, "R1")}
LOOKBACK = {"F": 2, "D": 1, "G": 2}
N_SIMS, SEED = 1000, 28
N_GUESS, N_REST = 3000, 1000                          # guesses per player; rest-of-season draws per player

B = SE.load_bundle(BUNDLE_DIR, TRADES_PATH)
K = B["stat_settings"]["k"]                           # notebook 27's scoring link
INPUTS = S.draw_inputs(pd.read_csv(DATA / "dev_residuals.csv"), pd.read_csv(DATA / "dev_ranges.csv"), center_pools=True)
FIT = INPUTS["fit"]                                   # notebook 17's luck fit: variance of a miss = a + b / minutes
keys = sorted(INPUTS["pools"])
KEY_ID = {k_: i for i, k_ in enumerate(keys)}
POOLS = [INPUTS["pools"][k_] for k_ in keys]
print(f"Scoring link k = {K}. Notebook 17's luck fit (WAR/60 units):")
print(FIT.round(5).to_string())
print("Talent spread sqrt(a): " + ", ".join(f"{g} {np.sqrt(FIT.loc[g, 'a']):.4f}" for g in FIT.index)
      + "; luck share at 1,000 minutes: " + ", ".join(f"{g} {FIT.loc[g, 'b'] / 1000 / (FIT.loc[g, 'a'] + FIT.loc[g, 'b'] / 1000):.0%}" for g in FIT.index))

Scoring link k = 0.18. Notebook 17's luck fit (WAR/60 units):
             a        b
group                  
F      0.00101  1.73365
D      0.00034  0.71322
G      0.00024  5.04322
Talent spread sqrt(a): F 0.0317, D 0.0183, G 0.0156; luck share at 1,000 minutes: F 63%, D 68%, G 95%


## 2. Evidence for past seasons: boxscores up to each team's 40th game

- **The download** (your approval): each game in 2013-14 to 2018-19 and 2021-22 to 2025-26 that is within the first 40 games of either team, saved in `data/schedule_raw/boxscores_past/`, which is gitignored. It's done once; finished games never change.
- **The cutoffs:** the dates when the **first** team reaches 10, 20 and 40 games. All games up to that date are evidence. That's how it works in real time: a date, not a game count.
- **The rest of a season** = MoneyPuck's season totals minus the evidence.
- **The check:** the boxscores' team goals must match the real scores of the same games.

In [2]:
skater_war = pd.read_csv(DATA / "war_by_season_2008_2025.csv")
goalie_war = pd.read_csv(DATA / "goalie_war_2008_2025.csv")
history = history_table(skater_war, goalie_war)
skaters, goalies = history[history["group"] != "G"], history[history["group"] == "G"]
bios = build_bio_table(DATA / "bio_raw", 2026)
raw = SC.scoring_history([DATA / "skaters_2008_to_2024.csv", DATA / "skaters25-26.csv"], skater_war)

old = schedule_games(DATA / "schedule_raw")
recent = S.games_from_shots(pd.read_parquet(DATA / "shots_2021-2025.parquet"))
games = pd.concat([old, recent], ignore_index=True)
results = S.add_results(games)
results["reg_goals"] = results["home_goals"] + results["away_goals"] - results["ot"]
results["gid"] = np.where(results["season"] >= 2021, results["season"].astype("int64") * 1000000 + results["game_id"].astype("int64"),
                          results["game_id"].astype("int64"))
real = S.standings(games)

BOX, CUT_DATE, EVID = {}, {}, {}
rows = []
for s_ in BT:
    b = boxscore_lines(DATA / "schedule_raw" / "boxscores_past" / str(s_))
    BOX[s_] = b
    tg = b.drop_duplicates(["game_id", "team"]).sort_values(["date", "game_id"])
    tg["n"] = tg.groupby("team").cumcount() + 1
    for n in CUTS:
        CUT_DATE[(s_, n)] = tg.loc[tg["n"] == n, "date"].min()
        EVID[(s_, n)] = set(b.loc[b["date"] <= CUT_DATE[(s_, n)], "game_id"])
    goals_box = b[b["group"] != "G"].groupby(["game_id", "team"])["goals"].sum()
    r = results[results["gid"].isin(b["game_id"])]
    so = r["last_period"] == "SO"
    real_goals = pd.concat([pd.Series((r["home_goals"] - (so & r["home_win"])).to_numpy(), index=pd.MultiIndex.from_arrays([r["gid"], r["home"]])),
                            pd.Series((r["away_goals"] - (so & ~r["home_win"])).to_numpy(), index=pd.MultiIndex.from_arrays([r["gid"], r["away"]]))])
    diff = goals_box.reindex(real_goals.index).fillna(0) - real_goals
    mism = (diff != 0).sum()
    rows.append({"season": label(s_), "boxscores": b["game_id"].nunique(), "games in the season": int((results["season"] == s_).sum()),
                 **{f"cutoff {n}": CUT_DATE[(s_, n)] for n in CUTS}, **{f"games by {n}": len(EVID[(s_, n)]) for n in CUTS},
                 "team-games off by a goal": int(mism), "boxscore has one more": int((diff == 1).sum()),
                 "boxscore has one fewer": int((diff == -1).sum())})
CHECK = pd.DataFrame(rows)
print(CHECK.to_string(index=False))
print(f"\nOff by a goal: {CHECK['team-games off by a goal'].sum()} team-games of about {2 * CHECK['boxscores'].sum():,}. "
      "In the shot-data seasons (2021-22 on) the boxscore almost always has one MORE goal: a goal missing from the shot "
      "data. The boxscore is the NHL's official record, and only boxscores are used as evidence.")

 season  boxscores  games in the season  cutoff 10  cutoff 20  cutoff 40  games by 10  games by 20  games by 40  team-games off by a goal  boxscore has one more  boxscore has one fewer
2013-14        612                 1230 2013-10-19 2013-11-10 2013-12-27          122          261          576                         1                      0                       1
2014-15        611                 1230 2014-10-25 2014-11-16 2015-01-02          116          266          566                         0                      0                       0
2015-16        608                 1230 2015-10-25 2015-11-16 2015-12-31          120          266          564                         0                      0                       0
2016-17        612                 1230 2016-10-30 2016-11-19 2017-01-02          130          270          563                         0                      0                       0
2017-18        631                 1271 2017-10-23 2017-11-14 2017-12-28   

## 3. Goalie evidence: goals saved above expected so far (2021-22 to 2025-26)

The raw MoneyPuck shot file has the goalie, the expected goal value and the result of every shot. From it comes each goalie's **GSAx in each game** (empty-net shots left out). Minutes come from the boxscores.

**The check:** a season's GSAx from the shots must match the goalie WAR table's season GSAx.

In [3]:
with zipfile.ZipFile(DATA / "shots_raw" / "shots_2021-2025.zip") as z_:
    with z_.open("shots_2021-2025.csv") as f_:
        shots = pd.read_csv(f_, usecols=["season", "game_id", "isPlayoffGame", "goalieIdForShot", "xGoal", "goal"])
shots = shots[(shots["isPlayoffGame"] == 0) & (shots["goalieIdForShot"].fillna(0) > 0)]
GG = shots.groupby(["season", "game_id", "goalieIdForShot"]).agg(xga=("xGoal", "sum"), ga=("goal", "sum")).reset_index()
GG["game_id"] = GG["season"].astype("int64") * 1000000 + GG["game_id"].astype("int64")
GG = GG.rename(columns={"goalieIdForShot": "playerId"}).astype({"playerId": "int64"})
GG["gsax"] = GG["xga"] - GG["ga"]
tot = GG.groupby(["season", "playerId"])["gsax"].sum()
j = goalie_war[goalie_war["season"].isin(GOALIE_SEASONS)].set_index(["season", "playerId"])["gsax"].to_frame("table").join(tot.rename("shots"), how="inner")
print(f"{len(j)} goalie-seasons: correlation {j['table'].corr(j['shots']):.4f}, average gap {(j['table'] - j['shots']).abs().mean():.2f} goals")
REPL = goalie_war.groupby("season")["repl_rate"].first()                  # replacement GSAx/60 per season

524 goalie-seasons: correlation 0.9972, average gap 0.40 goals


## 4. The two versions of a guess, and the checks that they behave

**For each player:** 3,000 guesses z from his notebook 17 pool.

- **Before any evidence**, the weights are all equal, so the posterior must equal the prior. That goes for both versions.
- **split:** the talent part is t = z x sqrt(a), where a is notebook 17's talent-change variance. With a full season still to play and no evidence, talent + fresh luck must have the same spread as the prior's z.

In [4]:
pid0 = [KEY_ID[k_] for k_ in keys if k_[0] == "F"][:4] + [KEY_ID[k_] for k_ in keys if k_[0] == "D"][:2]   # 4 forward, 2 defense pools
Z0 = IS.prior(np.array(pid0), POOLS, 20000, seed=3)
M = np.array([1400.0] * 6)
w0, ess0 = IS.weights(IS.log_lik_counts(np.zeros(6), np.full(6, 1.0), np.zeros(6), IS.scoring_factor(Z0, K)))
assert np.allclose(w0, 1 / 20000) and np.allclose(ess0, 20000)
for v in IS.VERSIONS:
    zr = IS.rest_z(Z0, w0, v, FIT, pd.Series(np.where(np.arange(6) < 4, "F", "D")), M, M, 20000, seed=4)
    print(f"{v:5s}: no evidence, full season left: spread of rest-of-season z / prior z = "
          + ", ".join(f"{x:.3f}" for x in zr.std(1) / Z0.std(1)))
print("No evidence: every weight equal, ESS = number of guesses (both versions).")

whole: no evidence, full season left: spread of rest-of-season z / prior z = 0.995, 0.996, 1.000, 1.002, 0.997, 1.002
split: no evidence, full season left: spread of rest-of-season z / prior z = 0.972, 0.988, 1.008, 1.015, 1.008, 1.020
No evidence: every weight equal, ESS = number of guesses (both versions).


## 5. Skaters: does the update predict the rest of the season better?

**For every skater** with 300+ minutes in a backtest season and a projection (notebook 27's goals/60 and assists/60, with the aging curve refit **without** that season), after 10, 20 and 40 games:

- **(a) preseason:** projected rate x his real rest-of-season minutes.
- **(b) pace:** his rate so far x his real rest-of-season minutes.
- **(c1) update, whole** and **(c2) update, split:** the weighted guesses.

**The evidence** is goals and assists so far: a Poisson count at rate x the guess's scoring factor x minutes.

**Measured:** the error of rest-of-season goals and points, and how often the real total lands in the 80% range. Real minutes are used, to isolate the rate, as notebook 27 did.

In [5]:
TG, TA = SC.rate_table(raw, "goals"), SC.rate_table(raw, "assists")

def skater_evidence(s_, n):
    """Per skater: real goals, assists and minutes up to the cutoff, and the rest of the season (season - so far)."""
    b = BOX[s_]
    b = b[(b["toi"] > 0) & (b["group"] != "G") & b["game_id"].isin(EVID[(s_, n)])]
    ev = b.groupby("playerId").agg(G=("goals", "sum"), A=("assists", "sum"), toi=("toi", "sum"))
    t_ = raw[raw["season"] == s_].set_index("playerId")[["group", "toi", "goals", "assists"]]
    p = ev.join(t_.rename(columns={"toi": "toi_season", "goals": "G_season", "assists": "A_season"}), how="inner")
    p["toi_rest"] = p["toi_season"] - p["toi"]
    p["G_rest"], p["A_rest"] = p["G_season"] - p["G"], p["A_season"] - p["A"]
    return p

def update_players(d, version, seed, war=None):
    """Weights for each player's guesses from goals and assists so far (and WAR so far if `war` is given:
    (observed WAR/60, projected WAR/60)). Returns (Z, w, ess, factor)."""
    Z = IS.prior(d["pool_id"].to_numpy(), POOLS, N_GUESS, seed=seed)
    ze = IS.evidence_z(Z, version, FIT, d["group"], d["toi_season"])
    f = IS.scoring_factor(ze, K)
    ll = IS.log_lik_counts(d["G"], d["g60"], d["toi"], f) + IS.log_lik_counts(d["A"], d["a60"], d["toi"], f)
    if war is not None:
        size = IS.draw_size(FIT, d["group"], d["toi_season"])
        level_z = Z if version == "whole" else ze
        ll = ll + IS.log_lik_war(war[0], war[1], level_z, size,
                                 IS.war_noise_var(version, FIT, d["group"], d["toi"], d["toi_season"]))
    w, ess = IS.weights(ll)
    return Z, w, ess, f

def rest_lines(d, Z, w, version, seed):
    """Rest-of-season goals and points: expected values and 80% ranges from the weighted guesses."""
    zr = IS.rest_z(Z, w, version, FIT, d["group"], d["toi_season"], d["toi_rest"], N_REST, seed=seed)
    zr0 = IS.rest_z(Z, np.full_like(w, 1 / Z.shape[1]), version, FIT, d["group"], d["toi_season"], d["toi_rest"], N_REST, seed=seed)
    m = np.exp(K * zr) / np.exp(K * zr0).mean(axis=1, keepdims=True)
    lg = d["g60"].to_numpy()[:, None] * m * d["toi_rest"].to_numpy()[:, None] / 60
    la = d["a60"].to_numpy()[:, None] * m * d["toi_rest"].to_numpy()[:, None] / 60
    rng = np.random.default_rng(seed + 1)
    G_, A_ = rng.poisson(lg), rng.poisson(la)
    return {"G": lg.mean(1), "A": la.mean(1), "G lo": np.percentile(G_, 10, axis=1), "G hi": np.percentile(G_, 90, axis=1),
            "P lo": np.percentile(G_ + A_, 10, axis=1), "P hi": np.percentile(G_ + A_, 90, axis=1)}

start = time.time()
parts = []
for s_ in BT:
    g60 = SC.project_rates(TG, bios, s_, 500, True, exclude_target=True)
    a60 = SC.project_rates(TA, bios, s_, 500, True, exclude_target=True)
    for n in CUTS:
        d = skater_evidence(s_, n)
        d["g60"], d["a60"] = g60.reindex(d.index).to_numpy(), a60.reindex(d.index).to_numpy()
        d = d[d["g60"].notna() & (d["toi_season"] >= 300) & (d["toi_rest"] >= 100)].reset_index()
        d["age"] = add_age(d[["playerId"]].assign(season=s_), bios)["age"].to_numpy()
        d["draft_overall"] = d["playerId"].map(bios.set_index("playerId")["draft_overall"])
        d["pool_id"] = [KEY_ID[b_] for b_ in S.draw_buckets(d["group"], d["age"], d["draft_overall"], INPUTS)]
        d["season"], d["cut"] = s_, n
        sd_ = s_ * 100 + n
        pre = rest_lines(d, IS.prior(d["pool_id"].to_numpy(), POOLS, N_GUESS, seed=sd_), np.full((len(d), N_GUESS), 1 / N_GUESS), "whole", sd_)
        for k_, v_ in pre.items():
            d[f"pre {k_}"] = v_
        for v in IS.VERSIONS:
            Z, w, ess, f = update_players(d, v, sd_)
            out = rest_lines(d, Z, w, v, sd_)
            for k_, v_ in out.items():
                d[f"{v} {k_}"] = v_
            d[f"{v} ess"] = ess
        parts.append(d)
SK = pd.concat(parts, ignore_index=True)
SK["P_rest"] = SK["G_rest"] + SK["A_rest"]
SK["pace G"] = SK["G"] / SK["toi"] * SK["toi_rest"]
SK["pace A"] = SK["A"] / SK["toi"] * SK["toi_rest"]
SK["period"] = np.where(SK["season"] <= TRAIN_END, "train", "test")
print(f"{len(SK):,} player-cutoffs in {time.time() - start:.0f} seconds")
print(SK.groupby(["period", "cut"]).size().unstack().to_string())

19,600 player-cutoffs in 41 seconds
cut       10    20    40
period                  
test    3555  3711  3763
train   2767  2886  2918


In [6]:
def sk_score(pred_g, pred_a, d, lo=None):
    e_g = np.sqrt(((pred_g - d["G_rest"]) ** 2).mean())
    e_p = np.sqrt(((pred_g + pred_a - d["P_rest"]) ** 2).mean())
    out = {"goals error": e_g, "points error": e_p}
    if lo is not None:
        out["goals in 80%"] = ((d["G_rest"] >= d[f"{lo} G lo"]) & (d["G_rest"] <= d[f"{lo} G hi"])).mean()
        out["points in 80%"] = ((d["P_rest"] >= d[f"{lo} P lo"]) & (d["P_rest"] <= d[f"{lo} P hi"])).mean()
    return out

rows = []
for (per, n), d in SK.groupby(["period", "cut"]):
    rows.append({"period": per, "cut": n, "model": "(a) preseason", **sk_score(d["pre G"], d["pre A"], d, "pre")})
    rows.append({"period": per, "cut": n, "model": "(b) pace", **sk_score(d["pace G"], d["pace A"], d)})
    for v, lab in [("whole", "(c1) update, whole"), ("split", "(c2) update, split")]:
        rows.append({"period": per, "cut": n, "model": lab, **sk_score(d[f"{v} G"], d[f"{v} A"], d, v)})
SKS = pd.DataFrame(rows)
for per in ["train", "test"]:
    print(f"{per.upper()} seasons ({', '.join(label(s_) for s_ in (BT_TRAIN if per == 'train' else BT_TEST))}): rest of season")
    print(SKS[SKS["period"] == per].drop(columns="period").round(3).to_string(index=False))
    print()
tr = SKS[(SKS["period"] == "train") & SKS["model"].str.startswith("(c")].groupby("model")[["points error", "goals error"]].mean()
VERSION = "split" if tr.loc["(c2) update, split", "points error"] <= tr.loc["(c1) update, whole", "points error"] else "whole"
te = SKS[(SKS["period"] == "test") & SKS["model"].str.startswith("(c")].groupby("model")[["points error", "goals error"]].mean()
best_te = "split" if te.loc["(c2) update, split", "points error"] <= te.loc["(c1) update, whole", "points error"] else "whole"
print(f"Version picked on training seasons (average points error over the cutoffs): {VERSION}; best on test: {best_te} "
      f"({'agrees' if best_te == VERSION else 'DISAGREES'})")

TRAIN seasons (2013-14, 2014-15, 2015-16, 2016-17, 2017-18): rest of season
 cut              model  goals error  points error  goals in 80%  points in 80%
  10      (a) preseason        4.028         7.277         0.824          0.819
  10           (b) pace       10.326        18.081           NaN            NaN
  10 (c1) update, whole        4.030         7.204         0.821          0.810
  10 (c2) update, split        4.010         7.181         0.827          0.827
  20      (a) preseason        3.622         6.542         0.836          0.818
  20           (b) pace        6.490        12.414           NaN            NaN
  20 (c1) update, whole        3.573         6.354         0.833          0.808
  20 (c2) update, split        3.574         6.363         0.842          0.834
  40      (a) preseason        2.897         5.190         0.855          0.832
  40           (b) pace        3.729         6.380           NaN            NaN
  40 (c1) update, whole        2.862        

**A blend with pace?** For each cutoff, weight x update + (1 - weight) x pace, with the weight fit on training seasons for points. If pace adds nothing, the weight comes out at 1.

**Adoption for skaters:** the update (or the blend) must beat **both** preseason and pace on the **test** seasons, at each cutoff.

In [7]:
ws = np.round(np.arange(0, 1.0001, 0.05), 2)
rows = []
BLEND = {}
for n in CUTS:
    tr_, te_ = SK[(SK["period"] == "train") & (SK["cut"] == n)], SK[(SK["period"] == "test") & (SK["cut"] == n)]
    pt = lambda d, w_: w_ * (d[f"{VERSION} G"] + d[f"{VERSION} A"]) + (1 - w_) * (d["pace G"] + d["pace A"])
    BLEND[n] = float(ws[int(np.argmin([np.sqrt(((pt(tr_, w_) - tr_["P_rest"]) ** 2).mean()) for w_ in ws]))])
    e = lambda p, d: np.sqrt(((p - d["P_rest"]) ** 2).mean())
    row = {"cut": n, "blend weight on the update (training)": BLEND[n],
           "test: preseason": e(te_["pre G"] + te_["pre A"], te_), "test: pace": e(te_["pace G"] + te_["pace A"], te_),
           "test: update": e(te_[f"{VERSION} G"] + te_[f"{VERSION} A"], te_), "test: blend": e(pt(te_, BLEND[n]), te_)}
    best_alt = min(row["test: preseason"], row["test: pace"])
    row["update wins"] = row["test: update"] < best_alt
    row["blend wins"] = row["test: blend"] < best_alt
    rows.append(row)
ADOPT_SK = pd.DataFrame(rows)
print(f"Rest-of-season points error on TEST seasons ({VERSION} version):")
print(ADOPT_SK.round(3).to_string(index=False))
SKATERS_ADOPTED = bool((ADOPT_SK["update wins"] | ADOPT_SK["blend wins"]).all())
print(f"\nSkater update adopted: {'YES' if SKATERS_ADOPTED else 'NO'} (it must win at every cutoff)")

Rest-of-season points error on TEST seasons (whole version):
 cut  blend weight on the update (training)  test: preseason  test: pace  test: update  test: blend  update wins  blend wins
  10                                    1.0            7.989      18.548         7.928        7.928         True        True
  20                                    1.0            7.177      11.847         6.958        6.958         True        True
  40                                    1.0            5.536       6.761         5.252        5.252         True        True

Skater update adopted: YES (it must win at every cutoff)


**Hockey sense: how far do updates move?** Small samples should move little, big ones more. Here are the biggest movers in the test seasons after 20 games: the updated scoring factor (1 = unchanged) and the effective sample size.

In [8]:
d = SK[(SK["period"] == "test") & (SK["cut"] == 20)].copy()
d["moved"] = (d[f"{VERSION} G"] + d[f"{VERSION} A"]) / (d["pre G"] + d["pre A"])
names = raw.drop_duplicates("playerId").set_index("playerId")["name"]
d["name"] = d["playerId"].map(names)
cols = ["name", "season", "G", "A", "toi", "g60", "a60", "moved", f"{VERSION} ess", "pre G", f"{VERSION} G", "pace G", "G_rest"]
print("Biggest risers (rest-of-season points vs. preseason):")
print(d.sort_values("moved", ascending=False).head(8)[cols].round(2).to_string(index=False))
print("\nBiggest fallers:")
print(d.sort_values("moved").head(8)[cols].round(2).to_string(index=False))
print(f"\nCorrelation of how far a player moved (|moved - 1|) with his minutes so far: {(d['moved'] - 1).abs().corr(d['toi']):.2f}")

Biggest risers (rest-of-season points vs. preseason):
           name  season  G  A    toi  g60  a60  moved  whole ess  pre G  whole G  pace G  G_rest
  Erik Karlsson    2022 11 17 480.58 0.36 1.11   1.66      59.15   9.59    15.94   37.08    14.0
     Troy Terry    2021 12 10 297.35 0.61 1.15   1.53     130.81  10.93    16.71   43.20    25.0
   Martin Necas    2024  9 16 268.65 0.99 1.45   1.48     170.81  20.24    29.90   41.28    18.0
Hampus Lindholm    2022  4 14 410.90 0.17 0.86   1.43     235.76   4.20     5.99   14.06     6.0
   Leo Carlsson    2025 11 15 347.47 0.93 1.21   1.39     389.07  15.47    21.55   31.44    18.0
  Morgan Rielly    2018  8 15 400.50 0.26 1.10   1.37     218.97   6.38     8.77   29.80    12.0
     Neal Pionk    2024  3 14 361.40 0.21 0.95   1.37     365.11   4.15     5.68    9.64     7.0
 Josh Morrissey    2022  1 15 347.63 0.27 0.87   1.36     448.11   7.01     9.51    4.44    15.0

Biggest fallers:
            name  season  G  A    toi  g60  a60  moved 

## 6. Goalies: does goalie evidence help? (2021-22 to 2025-26 only)

**The evidence:** each goalie's goals saved above expected so far, turned into WAR/60 the way `goalie_war` does it:

`(GSAx/60 - that season's replacement rate) / goals per win`

**The likelihood:** real WAR/60 so far = the guessed level + luck. The luck variance is b / minutes (split), or b x (1/minutes - 1/season minutes) (whole), with notebook 17's goalie luck fit.

**Predicting:** rest-of-season WAR/60, for goalies with a game before the cutoff and 300+ minutes after it.

**Said plainly:** 2013-14 to 2018-19 have no goalie test. These five seasons mix notebook 18's training and test eras.

In [9]:
rows = []
GPW = GOALS_PER_WIN
gproj = {}
for s_ in GOALIE_SEASONS:
    curve = aging_curve(history, bios, exclude_season=s_)[1]
    gproj[s_] = project(goalies, bios, s_, G_WEIGHTS, G_REG, AGING, curve)["proj_war_60"]
    tot_ = GG[GG["season"] == s_].groupby("playerId")["gsax"].sum()
    gtoi = goalie_war[goalie_war["season"] == s_].set_index("playerId")["toi"]
    b = BOX[s_]
    for n in CUTS:
        gb = b[(b["group"] == "G") & (b["toi"] > 0) & b["game_id"].isin(EVID[(s_, n)])].groupby("playerId")["toi"].sum()
        gs = GG[(GG["season"] == s_) & GG["game_id"].isin(EVID[(s_, n)])].groupby("playerId")["gsax"].sum()
        d = pd.DataFrame({"toi": gb}).join(gs.rename("gsax")).join(gtoi.rename("toi_season")).join(tot_.rename("gsax_season"))
        d["proj"] = gproj[s_].reindex(d.index)
        d = d.dropna()
        d["toi_rest"] = d["toi_season"] - d["toi"]
        d = d[d["toi_rest"] >= 300].reset_index()
        d["obs"] = (d["gsax"] / d["toi"] * 60 - REPL[s_]) / GPW
        d["rest_real"] = ((d["gsax_season"] - d["gsax"]) / d["toi_rest"] * 60 - REPL[s_]) / GPW
        d["season"], d["cut"], d["group"] = s_, n, "G"
        d["age"] = add_age(d[["playerId"]].assign(season=s_), bios)["age"].to_numpy()
        d["pool_id"] = [KEY_ID[b_] for b_ in S.draw_buckets(d["group"], d["age"], np.full(len(d), np.nan), INPUTS)]
        for v in IS.VERSIONS:
            Z = IS.prior(d["pool_id"].to_numpy(), POOLS, N_GUESS, seed=s_ * 100 + n)
            size = IS.draw_size(FIT, d["group"], d["toi_season"])
            level_z = Z if v == "whole" else IS.evidence_z(Z, v, FIT, d["group"], d["toi_season"])
            w, ess = IS.weights(IS.log_lik_war(d["obs"], d["proj"], level_z, size,
                                              IS.war_noise_var(v, FIT, d["group"], d["toi"], d["toi_season"])))
            zr = IS.rest_z(Z, w, v, FIT, d["group"], d["toi_season"], d["toi_rest"], N_REST, seed=s_ + n)
            lvl = d["proj"].to_numpy()[:, None] + zr * size[:, None]
            # the rest of the season's real WAR/60 also has its own luck: b / rest minutes (the split version's zr already has it)
            extra = 0 if v == "split" else np.sqrt(FIT.loc["G", "b"] / d["toi_rest"].to_numpy())[:, None] * np.random.default_rng(1).normal(size=lvl.shape)
            sim = lvl + extra
            d[f"{v}"] = lvl.mean(1)
            d[f"{v} lo"], d[f"{v} hi"] = np.percentile(sim, 10, axis=1), np.percentile(sim, 90, axis=1)
            d[f"{v} ess"] = ess
        Z = IS.prior(d["pool_id"].to_numpy(), POOLS, N_GUESS, seed=s_ * 100 + n)
        size = IS.draw_size(FIT, d["group"], d["toi_season"])
        pre = d["proj"].to_numpy()[:, None] + Z * size[:, None] + np.sqrt(FIT.loc["G", "b"] / d["toi_rest"].to_numpy())[:, None] * np.random.default_rng(2).normal(size=Z.shape)
        d["pre lo"], d["pre hi"] = np.percentile(pre, 10, axis=1), np.percentile(pre, 90, axis=1)
        rows.append(d)
GK = pd.concat(rows, ignore_index=True)
werr = lambda p, d: float(np.sqrt(np.average((p - d["rest_real"]) ** 2, weights=d["toi_rest"])))
out = []
for n, d in GK.groupby("cut"):
    row = {"cut": n, "goalies": len(d), "preseason": werr(d["proj"], d), "pace": werr(d["obs"], d)}
    for v in IS.VERSIONS:
        row[f"update ({v})"] = werr(d[v], d)
        row[f"{v} in 80%"] = ((d["rest_real"] >= d[f"{v} lo"]) & (d["rest_real"] <= d[f"{v} hi"])).mean()
    row["preseason in 80%"] = ((d["rest_real"] >= d["pre lo"]) & (d["rest_real"] <= d["pre hi"])).mean()
    out.append(row)
GKS = pd.DataFrame(out)
print("Goalies, rest-of-season WAR/60 error (minutes-weighted), 2021-22 to 2025-26:")
print(GKS.round(4).to_string(index=False))
GOALIES_ADOPTED = bool((GKS[f"update ({VERSION})"] < np.minimum(GKS["preseason"], GKS["pace"])).all())
print(f"\nGoalie update ({VERSION} version) beats preseason and pace at every cutoff: {'YES' if GOALIES_ADOPTED else 'NO'}")

Goalies, rest-of-season WAR/60 error (minutes-weighted), 2021-22 to 2025-26:
 cut  goalies  preseason   pace  update (whole)  whole in 80%  update (split)  split in 80%  preseason in 80%
  10      317     0.0559 0.1661          0.0561        0.8770          0.0558        0.7886            0.8927
  20      329     0.0587 0.1174          0.0588        0.8815          0.0584        0.7660            0.8875
  40      328     0.0671 0.0995          0.0697        0.8079          0.0667        0.7774            0.8720

Goalie update (whole version) beats preseason and pace at every cutoff: NO


## 7. Teams: rest-of-season points

**The setup** (each backtest season and cutoff, the season's real remaining schedule): notebook 26's past lineups, all-seasons fits and team bonus (notebook 25's residual).

**Compared:**

- **(a) preseason:** the simulator as it was before the season.
- **(b) pace:** points per game so far x games left.
- **(c) update:** players updated (goals and assists), plus the team bonus updated with what's left.
- **(c+g):** (c) plus the goalie update (2021-22 to 2025-26 only).

**The team bonus update:**

- **The gap:** real points so far minus what the updated players would have expected in those games.
  - That expectation is the preseason expectation so far, plus the per-game change the player update makes going forward, times games played.
- **The weight:** the gap gets w = g / (g + g0).
- **What g0 is:**
  - (the luck variance of a team's points per game) / (the variance of real team effects);
  - both are measured from these seasons: luck from single games, team effects from how much one season's residual carries to the next.

In [10]:
SEASONS = sorted(set(TARGETS) | {2012, 2020})              # past lineups: every target + the short "last seasons"
f_train, m_train = training_nhle(pd.read_csv(DATA / "career_totals.csv"), TRAIN_END)
careers = pd.read_csv(DATA / "career_totals.csv")
profiles = load_profiles(DATA / "bio_raw")
nhle_train = best_nhle(careers, f_train, m_train, rule="last2")
pairs = pd.concat([candidates(history, SEASONS + [2026]),
                   skaters[skaters["season"].isin(SEASONS)][["playerId", "season", "group"]]]).drop_duplicates(["playerId", "season"])
feat = skater_features(pairs, history, bios, profiles, nhle_train)
model = fit_prior(fit_rows(history, feat, TRAIN), PRIOR_INPUTS)
cand = candidates(history, SEASONS).merge(feat.drop(columns="group"), on=["playerId", "season"], how="left")
priors = cand[["playerId", "season"]].assign(prior_gar_60=apply_prior(cand, model).to_numpy())
rows = []
for t in SEASONS:
    curve = aging_curve(history, bios, exclude_season=t)[1]
    ps = project(skaters, bios, t, SK_WEIGHTS, SK_REG, AGING, curve, priors=priors, prior_reg=1500)
    pg = project(goalies, bios, t, G_WEIGHTS, G_REG, AGING, curve)
    rows.append(pd.concat([ps, pg])[["proj_war_60"]].assign(season=t).rename_axis("playerId").reset_index())
bt_proj = pd.concat(rows, ignore_index=True)
faceoffs = L.faceoff_rates([DATA / "skaters_2008_to_2024.csv", DATA / "skaters25-26.csv"])
usage = L.usage_table(skater_war, goalie_war, faceoffs)
exp = experience(history, profiles, history[["playerId", "season"]])

def make_pool(t):
    """Players with 1+ games for a team in season t, with projection and usage (notebook 24's main pool)."""
    u = usage[(usage["season"] == t) & (usage["games_played"] >= 1)]
    p = (u[["playerId", "name", "team", "position", "group", "games_played", "usage"]]
         .rename(columns={"games_played": "gp", "usage": "actual"}).reset_index(drop=True))
    p["proj_war_60"] = p["playerId"].map(bt_proj[bt_proj["season"] == t].set_index("playerId")["proj_war_60"])
    miss = p["proj_war_60"].isna() & (p["group"] != "G")
    f = p.loc[miss, ["playerId", "group"]].merge(feat[feat["season"] == t].drop(columns="group"), on="playerId", how="left")
    p.loc[miss, "proj_war_60"] = (apply_prior(f, model) / GOALS_PER_WIN).to_numpy()
    still = p["proj_war_60"].isna()
    p.loc[still, "proj_war_60"] = p.loc[still, "group"].map(group_average(history, t - 1) / GOALS_PER_WIN)
    p["usage_last"] = L.usage_by_group(usage, p, t, LOOKBACK)
    p["natural_center"] = L.natural_center(usage, p["playerId"], p["position"], t) & (p["group"] == "F")
    g = p["group"] == "G"
    p.loc[g, "actual"] = p.loc[g, "gp"] / p[g].groupby("team")["gp"].transform("sum")
    p["season"] = t
    return p

lineups = {}
for t in SEASONS:
    seasons_ = [t - 3, t - 2, t - 1]
    defaults, slots = L.rookie_defaults(usage, exp, seasons_), L.slot_minutes(usage, seasons_)
    share, totals = L.starter_share(usage, seasons_)[1], L.team_minutes(usage, seasons_)
    sc = L.coach_scores(make_pool(t), CHOSEN, defaults)
    lu = pd.concat([L.pick_lineup(x) for _, x in sc.groupby("team")])
    lu["team"] = [nhl_code(x, t) for x in lu["team"]]
    lu["minutes_per_game"] = L.assign_minutes(lu, L.scaled_slots(slots, totals), share, own_weight=SE.OWN_WEIGHT)
    lu["age"] = add_age(lu[["playerId"]].assign(season=t), bios)["age"].to_numpy()
    lu["draft_overall"] = lu["playerId"].map(bios.set_index("playerId")["draft_overall"])
    lineups[t] = lu.reset_index(drop=True)
rows = []
for t in SEASONS:
    teams_, sk, gk = S.team_strength(lineups[t])
    rows.append(pd.DataFrame({"season": t, "team": teams_, "skater": sk[0], "goalie": gk[0]}))
STRENGTH = pd.concat(rows, ignore_index=True)
s = STRENGTH.set_index(["season", "team"])
GAMES = results[results["season"].isin(TARGETS)].copy()
h = s.reindex(pd.MultiIndex.from_arrays([GAMES["season"], GAMES["home"]]))
a = s.reindex(pd.MultiIndex.from_arrays([GAMES["season"], GAMES["away"]]))
GAMES["ds"] = h["skater"].to_numpy() - a["skater"].to_numpy()
GAMES["dg"] = h["goalie"].to_numpy() - a["goalie"].to_numpy()
LATE = S.late_table(recent)
reg_pg = results.groupby("season")["reg_goals"].mean()

def normal_total(seasons_):
    return float(reg_pg.reindex(seasons_).dropna().mean() - LATE["extra_goals"])

def fit_all(t):
    """Notebook 25's "all seasons" fit for target season t: every target season before it."""
    seasons_ = [x for x in TARGETS if x < t]
    return S.fit_game_model(GAMES[GAMES["season"].isin(seasons_)], LATE, normal_total(seasons_))

def settings_for(fit, t):
    return S.game_settings(fit, LATE, normal_total([t - 3, t - 2, t - 1]))

def pool_ids(lu):
    return np.array([KEY_ID[b_] for b_ in S.draw_buckets(lu["group"], lu["age"], lu["draft_overall"], INPUTS)])

def simulate_games_of(t, sched, Z, bonus, fit, seed):
    """Average simulated points per team over N_SIMS seasons of `sched`, with each lineup row's z in Z."""
    lu = lineups[t]
    gp = pd.concat([results.loc[results["season"] == t, "home"], results.loc[results["season"] == t, "away"]]).value_counts()
    size = IS.draw_size(FIT, lu["group"], lu["minutes_per_game"] * lu["team"].map(gp).to_numpy())
    table, _ = S.simulate_season(lu, sched[["home", "away"]], settings_for(fit, t), N_SIMS, seed=seed,
                                 draws=Z * size[:, None], bonus=bonus)
    return table.set_index("team")["points"]

# Notebook 25's residual bonus for each backtest season (residual of the season before, lambda 0.4)
REAL = real[["season", "team", "gp", "points"]].rename(columns={"points": "real"})
MOVED = {("ATL", 2011): "WPG", ("PHX", 2014): "ARI", ("ARI", 2024): "UTA"}
def full_season(t, fit):
    sched = results[results["season"] == t]
    lu = lineups[t]
    Z = IS.prior(pool_ids(lu), POOLS, N_SIMS, seed=t)
    return simulate_games_of(t, sched, Z, None, fit, seed=t)
FITS = {t: fit_all(t) for t in sorted(set(BT) | set(range(2013, 2018)))}
d = pd.concat([full_season(t, FITS[t]).rename("points").reset_index().assign(season=t) for t in [2013, 2014, 2015, 2016, 2017]])
d = d.merge(REAL, on=["season", "team"]).merge(STRENGTH, on=["season", "team"])
d["q"] = [FITS[t]["a_s"] * s_ + FITS[t]["a_g"] * g_ for t, s_, g_ in zip(d["season"], d["skater"], d["goalie"])]
d["q"] -= d.groupby("season")["q"].transform("mean")
BETA = float(np.polyfit(d["q"], d["points"] / d["gp"], 1)[0])
BONUS = {}
for t in BT:
    prev = full_season(t - 1, FITS[t]).rename("points").reset_index().assign(season=t - 1).merge(REAL, on=["season", "team"])
    r = (prev["real"] - prev["points"]) / prev["gp"]
    resid = pd.Series((r - r.mean()).to_numpy(), index=[MOVED.get((tm, t), tm) for tm in prev["team"]])
    teams_t = sorted(lineups[t]["team"].unique())
    BONUS[t] = (SE.TEAM_LAMBDA * resid.reindex(teams_t).fillna(0.0) / BETA).to_dict()
print(f"Past lineups for {len(SEASONS)} seasons; beta {BETA:.3f} (notebook 25: 0.379); team bonuses for {len(BONUS)} backtest seasons")

Past lineups for 16 seasons; beta 0.379 (notebook 25: 0.379); team bonuses for 11 backtest seasons


In [11]:
G60 = {s_: SC.project_rates(TG, bios, s_, 500, True, exclude_target=True) for s_ in BT}
A60 = {s_: SC.project_rates(TA, bios, s_, 500, True, exclude_target=True) for s_ in BT}

def lineup_update(t, n, version, use_goalies, seed):
    """Rest-of-season z for every lineup row of season t after cutoff n (no evidence: equal weights)."""
    lu = lineups[t]
    sched = results[results["season"] == t]
    ev = sched["gid"].isin(EVID[(t, n)])
    gp_all = pd.concat([sched["home"], sched["away"]]).value_counts()
    gp_rest = pd.concat([sched.loc[~ev, "home"], sched.loc[~ev, "away"]]).value_counts()
    d = lu[["playerId", "team", "group", "minutes_per_game"]].copy()
    d["toi_season"] = (d["minutes_per_game"] * d["team"].map(gp_all)).clip(lower=1.0)
    d["toi_rest"] = (d["minutes_per_game"] * d["team"].map(gp_rest).fillna(0)).clip(lower=1.0)
    b = BOX[t][(BOX[t]["toi"] > 0) & BOX[t]["game_id"].isin(EVID[(t, n)])]
    ev_sk = b[b["group"] != "G"].groupby("playerId").agg(G=("goals", "sum"), A=("assists", "sum"), toi=("toi", "sum"))
    d = d.join(ev_sk, on="playerId")
    d[["G", "A", "toi"]] = d[["G", "A", "toi"]].fillna(0.0)
    d["g60"] = d["playerId"].map(G60[t]).fillna(0.0)
    d["a60"] = d["playerId"].map(A60[t]).fillna(0.0)
    has_rate = ((d["g60"] > 0) & (d["group"] != "G")).to_numpy()
    Z = IS.prior(pool_ids(lu), POOLS, N_GUESS, seed=seed)
    ze = IS.evidence_z(Z, version, FIT, d["group"], d["toi_season"])
    f = IS.scoring_factor(ze, K)
    ll = IS.log_lik_counts(d["G"], d["g60"], d["toi"], f) + IS.log_lik_counts(d["A"], d["a60"], d["toi"], f)
    ll = np.where(has_rate[:, None], ll, 0.0)
    if use_goalies and t in GOALIE_SEASONS:
        gt = b[b["group"] == "G"].groupby("playerId")["toi"].sum()
        gs = GG[(GG["season"] == t) & GG["game_id"].isin(EVID[(t, n)])].groupby("playerId")["gsax"].sum()
        gtoi = d["playerId"].map(gt)
        obs = ((d["playerId"].map(gs).fillna(0) / gtoi * 60 - REPL[t]) / GOALS_PER_WIN).where((d["group"] == "G") & (gtoi > 0))
        level_z = Z if version == "whole" else ze
        ll = ll + IS.log_lik_war(obs, lu["proj_war_60"], level_z, IS.draw_size(FIT, d["group"], d["toi_season"]),
                                 IS.war_noise_var(version, FIT, d["group"], gtoi.fillna(1.0), d["toi_season"]))
    w, _ = IS.weights(ll)
    return IS.rest_z(Z, w, version, FIT, d["group"], d["toi_season"], d["toi_rest"], N_SIMS, seed=seed + 1)

# g0 for the team update: (luck variance of a team's points per game) / (variance of real team effects),
# both from the training seasons. Team effects = what's left after the preseason model, minus its luck part.
team_games = pd.concat([pd.DataFrame({"season": results["season"], "team": results["home"], "pts": results["home_points"]}),
                        pd.DataFrame({"season": results["season"], "team": results["away"], "pts": results["away_points"]})])
tg_ = team_games[team_games["season"].isin(BT_TRAIN)]
SIGMA2 = float(tg_.groupby(["season", "team"])["pts"].var().mean())
rows = []
for t in BT_TRAIN:
    pf = simulate_games_of(t, results[results["season"] == t], IS.prior(pool_ids(lineups[t]), POOLS, N_SIMS, seed=t),
                           BONUS[t], FITS[t], seed=t).rename("points").reset_index().assign(season=t).merge(REAL, on=["season", "team"])
    rows.append(pf.assign(r=(pf["real"] - pf["points"]) / pf["gp"]))
RF = pd.concat(rows)
RF["r"] -= RF.groupby("season")["r"].transform("mean")
TAU2 = float(RF["r"].var() - SIGMA2 / RF["gp"].mean())
G0 = SIGMA2 / TAU2
print(f"Luck variance of a team's points per game: {SIGMA2:.3f}; variance of real team effects left after the preseason "
      f"model: {TAU2:.4f} (points per game squared) -> g0 = {G0:.0f} games")
print("So a team's leftover results count " + ", ".join(f"{IS.team_weight(g, G0):.0%} after {g} games" for g in [10, 20, 40]))

start = time.time()
rows = []
for t in BT:
    sched = results[results["season"] == t]
    lu, fit = lineups[t], FITS[t]
    for n in CUTS:
        ev = sched["gid"].isin(EVID[(t, n)])
        so_far, rest = sched[ev], sched[~ev]
        sd_ = t * 100 + n
        Z0 = IS.prior(pool_ids(lu), POOLS, N_SIMS, seed=sd_)
        pre_rest = simulate_games_of(t, rest, Z0, BONUS[t], fit, sd_)
        pre_sofar = simulate_games_of(t, so_far, Z0, BONUS[t], fit, sd_).reindex(pre_rest.index)
        st = S.standings(so_far[list(games.columns)]).set_index("team").reindex(pre_rest.index).fillna(0)
        gp_rest = pd.concat([rest["home"], rest["away"]]).value_counts().reindex(pre_rest.index)
        real_rest = REAL[REAL["season"] == t].set_index("team")["real"].reindex(pre_rest.index) - st["points"]
        out = pd.DataFrame({"season": t, "cut": n, "gp so far": st["gp"], "real rest": real_rest,
                            "(a) preseason": pre_rest, "(b) pace": st["points"] / st["gp"] * gp_rest})
        for tag, use_g in [("(c) update", False), ("(c+g) update + goalies", True)]:
            if use_g and t not in GOALIE_SEASONS:
                continue
            Zr = lineup_update(t, n, VERSION, use_g, sd_)
            upd_rest = simulate_games_of(t, rest, Zr, BONUS[t], fit, sd_)
            expect_sofar = pre_sofar + (upd_rest - pre_rest) / gp_rest * st["gp"]
            gap = (st["points"] - expect_sofar) / st["gp"]
            new_bonus = pd.Series(BONUS[t]).reindex(pre_rest.index).fillna(0) + IS.team_weight(st["gp"], G0) * gap / BETA
            out[f"{tag}: players only"] = upd_rest
            out[tag] = simulate_games_of(t, rest, Zr, new_bonus.to_dict(), fit, sd_)
        rows.append(out.rename_axis("team").reset_index())
    print(label(t), f"{time.time() - start:.0f}s", flush=True)
TB = pd.concat(rows, ignore_index=True)
TB["period"] = np.where(TB["season"] <= TRAIN_END, "train", "test")
models = ["(a) preseason", "(b) pace", "(c) update: players only", "(c) update", "(c+g) update + goalies: players only", "(c+g) update + goalies"]
rows = []
for (per, n), d in TB.groupby(["period", "cut"]):
    for m in models:
        x = d[d[m].notna()] if m in d else d.iloc[0:0]
        if len(x) == 0:
            continue
        rows.append({"period": per, "cut": n, "model": m, "teams": len(x),
                     "error (points)": np.sqrt(((x[m] - x["real rest"]) ** 2).mean()),
                     "correlation": x.groupby("season").apply(lambda y: y[m].corr(y["real rest"]), include_groups=False).mean()})
TBS = pd.DataFrame(rows)
for per in ["train", "test"]:
    print(f"\n{per.upper()} seasons: team rest-of-season points")
    print(TBS[TBS["period"] == per].drop(columns="period").round(3).to_string(index=False))
g_only = TB[TB["season"].isin(GOALIE_SEASONS)]
print("\nGoalie seasons only (2021-22 to 2025-26), with and without the goalie update:")
print(g_only.groupby("cut").apply(lambda d: pd.Series({m: np.sqrt(((d[m] - d["real rest"]) ** 2).mean())
                                                       for m in ["(a) preseason", "(b) pace", "(c) update", "(c+g) update + goalies"]}),
                                  include_groups=False).round(3).to_string())
te = TBS[TBS["period"] == "test"].pivot(index="cut", columns="model", values="error (points)")
TEAMS_ADOPTED = bool((te["(c) update"] < np.minimum(te["(a) preseason"], te["(b) pace"])).all())
print(f"\nTeam update (players + team bonus) beats preseason and pace on test seasons at every cutoff: {'YES' if TEAMS_ADOPTED else 'NO'}")

Luck variance of a team's points per game: 0.845; variance of real team effects left after the preseason model: 0.0172 (points per game squared) -> g0 = 49 games
So a team's leftover results count 17% after 10 games, 29% after 20 games, 45% after 40 games
2013-14 4s
2014-15 8s
2015-16 12s
2016-17 15s
2017-18 19s
2018-19 23s
2021-22 30s
2022-23 37s
2023-24 44s
2024-25 51s
2025-26 58s

TRAIN seasons: team rest-of-season points
 cut                    model  teams  error (points)  correlation
  10            (a) preseason    151          12.450        0.402
  10                 (b) pace    151          25.574        0.372
  10 (c) update: players only    151          12.278        0.425
  10               (c) update    151          11.956        0.480
  20            (a) preseason    151          11.508        0.388
  20                 (b) pace    151          16.234        0.439
  20 (c) update: players only    151          11.266        0.422
  20               (c) update    151       

## 8. The verdict: what's adopted

An update is used for 2026-27 only if it, or its blend with pace, beats **both** the preseason projection and the pace on the **test** seasons:

- **Skaters:** the goals and assists update.
- **Goalies:** the goals-saved-above-expected update (2021-22 to 2025-26).
- **Teams:** the player update plus the team bonus update.

In [12]:
ADOPT = {"version": VERSION, "skaters": SKATERS_ADOPTED, "goalies": GOALIES_ADOPTED, "teams": TEAMS_ADOPTED}
print(f"Version of a guess: {VERSION}")
print(f"Skater update (goals, assists): {'ADOPTED' if SKATERS_ADOPTED else 'not adopted'}")
print(f"Goalie update (GSAx): {'ADOPTED' if GOALIES_ADOPTED else 'not adopted'}")
print(f"Team bonus update: {'ADOPTED' if TEAMS_ADOPTED else 'not adopted'}")
print("Skater WAR evidence: not testable; shown as a separate version, never the default")

Version of a guess: whole
Skater update (goals, assists): ADOPTED
Goalie update (GSAx): not adopted
Team bonus update: ADOPTED
Skater WAR evidence: not testable; shown as a separate version, never the default


## 9. 2026-27: updating with this season's games

**The evidence through Nov 9:**

- **Goals, assists and minutes:** the boxscores (notebook 27's `real_games.csv`).
- **Goalies' GSAx:** MoneyPuck's 2026-27 goalie file, through the same date, and with your download.
- **Skater WAR so far:** our WAR model run on MoneyPuck's 2026-27 skater file. It's used **only** in the second version.

**Reality mode** (notebook 26): from today, trades through today, Larkin and Edvinsson sitting.

**Each player's minutes** come from that lineup:

- full season = minutes per game x 84;
- rest of the season = minutes per game x his team's games left.

In [13]:
SCHED26, STAMP = season_schedule(DATA / "schedule_raw" / "2026-27")
TODAY = STAMP[:10]
pl = B["players"]
pid = lambda name: int(pl.loc[pl["name"] == name, "playerId"].iloc[0])
SIT = [pid("Dylan Larkin"), pid("Simon Edvinsson")]
lu26 = SE.scenario_lineups(B, SE.default_trades(B["trades"], TODAY), SIT)
left = pd.concat([SCHED26.loc[SCHED26["date"] >= TODAY, "home"], SCHED26.loc[SCHED26["date"] >= TODAY, "away"]]).value_counts()
P26 = pl[pl["in_pool"]].sort_values("playerId")[["playerId", "name", "team", "group", "proj_war_60", "pool_id"]].reset_index(drop=True)
P26 = P26.merge(lu26[["playerId", "team", "minutes_per_game", "dressed"]].rename(columns={"team": "team_now"}), on="playerId", how="left")
rg = B["real_games"]
rg = rg[(rg["date"] < TODAY) & (rg["toi"] > 0)]
ev26 = rg[rg["group"] != "G"].groupby("playerId").agg(G=("goals", "sum"), A=("assists", "sum"), toi=("toi", "sum"), gp=("game_id", "size"))
P26 = P26.join(ev26, on="playerId")
P26[["G", "A", "toi", "gp"]] = P26[["G", "A", "toi", "gp"]].fillna(0.0)
P26["g60"] = P26["playerId"].map(B["scoring"]["g60"]).fillna(0.0)
P26["a60"] = P26["playerId"].map(B["scoring"]["a60"]).fillna(0.0)
mpg = P26["minutes_per_game"].fillna(0.0)
P26["toi_season"] = np.maximum(mpg * SE.GAMES_PER_TEAM, P26["toi"]).clip(lower=1.0)
P26["toi_rest"] = (mpg * P26["team_now"].map(left).fillna(0)).clip(lower=1.0)
war26 = run_season(DATA / "skaters26-27.csv").set_index("playerId")
P26["war60_so_far"] = (P26["playerId"].map(war26["war"]) / P26["playerId"].map(war26["all_toi"]) * 60).where(P26["group"] != "G")
P26["war_min"] = P26["playerId"].map(war26["all_toi"])
g26 = load_goalies(pd.read_csv(DATA / "goalies26-27.csv")).groupby("playerId").agg(gsax=("gsax", "sum"), gtoi=("toi", "sum"))
league26 = g26["gsax"].sum() / g26["gtoi"].sum() * 60
repl26 = league26 + float((goalie_war.groupby("season").apply(lambda d: d["repl_rate"].iloc[0] - d["gsax"].sum() / d["toi"].sum() * 60, include_groups=False)).mean())
P26["gwar60_so_far"] = ((P26["playerId"].map(g26["gsax"]) / P26["playerId"].map(g26["gtoi"]) * 60 - repl26) / GOALS_PER_WIN).where(P26["group"] == "G")
P26["gmin"] = P26["playerId"].map(g26["gtoi"])
print(f"Evidence through {rg['date'].max()}: {len(ev26)} skaters with games; skater WAR so far for {P26['war60_so_far'].notna().sum()}, "
      f"goalie GSAx for {P26['gwar60_so_far'].notna().sum()} goalies (2026-27 replacement rate {repl26:.3f} GSAx/60)")

def update26(with_skater_war, seed=26):
    """Weights and rest-of-season z for every 2026-27 pool player (the adopted version and evidence)."""
    Z = IS.prior(P26["pool_id"].to_numpy(), POOLS, N_GUESS, seed=seed)
    ze = IS.evidence_z(Z, VERSION, FIT, P26["group"], P26["toi_season"])
    f = IS.scoring_factor(ze, K)
    size = IS.draw_size(FIT, P26["group"], P26["toi_season"])
    level_z = Z if VERSION == "whole" else ze
    ll = np.zeros_like(Z)
    if SKATERS_ADOPTED:
        sk = ((P26["group"] != "G") & (P26["g60"] > 0)).to_numpy()[:, None]
        ll += np.where(sk, IS.log_lik_counts(P26["G"], P26["g60"], P26["toi"], f) + IS.log_lik_counts(P26["A"], P26["a60"], P26["toi"], f), 0.0)
    if GOALIES_ADOPTED:
        ll += IS.log_lik_war(P26["gwar60_so_far"], P26["proj_war_60"], level_z, size,
                             IS.war_noise_var(VERSION, FIT, P26["group"], P26["gmin"].fillna(1.0), P26["toi_season"]))
    if with_skater_war:
        ll += IS.log_lik_war(P26["war60_so_far"], P26["proj_war_60"], level_z, size,
                             IS.war_noise_var(VERSION, FIT, P26["group"], P26["war_min"].fillna(1.0), P26["toi_season"]))
    w, ess = IS.weights(ll)
    zr = IS.rest_z(Z, w, VERSION, FIT, P26["group"], P26["toi_season"], P26["toi_rest"], 4000, seed=seed + 1)
    zr0 = IS.rest_z(Z, np.full_like(w, 1 / Z.shape[1]), VERSION, FIT, P26["group"], P26["toi_season"], P26["toi_rest"], 4000, seed=seed + 1)
    norm = np.exp(K * zr0).mean(axis=1)
    out = pd.DataFrame({"playerId": P26["playerId"], "ess": ess,
                        "war60": P26["proj_war_60"] + IS.posterior_mean(level_z, w) * size,
                        "factor": np.exp(K * zr).mean(axis=1) / norm})
    out["g60"], out["a60"] = P26["g60"] * out["factor"], P26["a60"] * out["factor"]
    return out, zr

U_DEF, Z_DEF = update26(False)

def moved_after(name, k):
    """A player's updated points/60 after his first k games this season (goals and assists only), and the ESS."""
    i = pid(name)
    r_ = rg[rg["playerId"] == i].sort_values(["date", "game_id"]).head(k)
    d_ = P26[P26["playerId"] == i].copy()
    d_["G"], d_["A"], d_["toi"] = r_["goals"].sum(), r_["assists"].sum(), r_["toi"].sum()
    Z = IS.prior(d_["pool_id"].to_numpy(), POOLS, N_GUESS, seed=7)
    f = IS.scoring_factor(IS.evidence_z(Z, VERSION, FIT, d_["group"], d_["toi_season"]), K)
    w, ess = IS.weights(IS.log_lik_counts(d_["G"], d_["g60"], d_["toi"], f) + IS.log_lik_counts(d_["A"], d_["a60"], d_["toi"], f))
    pre = float(d_["g60"].iloc[0] + d_["a60"].iloc[0])
    return {"player": name, "games": len(r_), "G": int(d_["G"].iloc[0]), "A": int(d_["A"].iloc[0]),
            "minutes": float(d_["toi"].iloc[0]), "P/60 preseason": pre, "P/60 updated": pre * float(IS.posterior_mean(f, w)[0]),
            "moved": float(IS.posterior_mean(f, w)[0]) - 1, "ESS": float(ess[0])}

rows = [moved_after(nm, k) for nm in ["Cutter Gauthier", "Connor McDavid", "Leon Draisaitl", "Adam Fantilli"] for k in [6, 99]]
print("Small samples move little, bigger ones more (the update after a player's first 6 games vs. all his games so far):")
print(pd.DataFrame(rows).round(3).to_string(index=False))
U_WAR, Z_WAR = update26(True)
print("Updated: default and with skater WAR evidence; median effective sample size "
      f"{np.median(U_DEF['ess']):.0f} and {np.median(U_WAR['ess']):.0f} of {N_GUESS:,}")

Evidence through 2026-11-09: 758 skaters with games; skater WAR so far for 755, goalie GSAx for 74 goalies (2026-27 replacement rate -0.244 GSAx/60)
Small samples move little, bigger ones more (the update after a player's first 6 games vs. all his games so far):
         player  games  G  A  minutes  P/60 preseason  P/60 updated  moved      ESS
Cutter Gauthier      6  2  3  104.817           2.743         2.744  0.001 2951.215
Cutter Gauthier     17 15  7  311.117           2.743         3.409  0.243 1266.912
 Connor McDavid      6  4 13  141.383           4.175         4.987  0.194 1311.541
 Connor McDavid     19 11 37  450.583           4.175         5.433  0.301  608.940
 Leon Draisaitl      6  4  7  127.000           3.742         4.006  0.070 2539.394
 Leon Draisaitl     19 18 18  416.833           3.742         4.375  0.169 1396.806
  Adam Fantilli      6  1  1  122.367           2.429         2.172 -0.106 2384.876
  Adam Fantilli     15  4  2  312.300           2.429         2.0

In [14]:
# Team bonus update for 2026-27 (if adopted): real points so far vs. what the preseason model and the updated players expect
BASE = SE.run_scenario(B, n_sims=10000, seed=SEED)                            # opening night, preseason
PRE = SE.run_scenario(B, start_date=TODAY, sit_out=SIT, n_sims=10000, seed=SEED, stat_lines=True)
UPD_P = SE.run_scenario(B, start_date=TODAY, sit_out=SIT, n_sims=10000, seed=SEED,
                        updated={"playerId": P26["playerId"].to_numpy(), "z": Z_DEF})
real26 = PRE["table"].set_index("team")
gp_left = real26.index.map(left).to_numpy()
pre_rest = real26["points"] - real26["real_points"]
upd_rest = UPD_P["table"].set_index("team")["points"] - real26["real_points"]
base_ppg = BASE["table"].set_index("team")["points"] / SE.GAMES_PER_TEAM
expect_sofar = (base_ppg + (upd_rest - pre_rest) / gp_left) * real26["real_gp"]
gap = (real26["real_points"] - expect_sofar) / real26["real_gp"]
bonus0 = pd.Series(B["bonus"]).reindex(real26.index)
BONUS26 = (bonus0 + IS.team_weight(real26["real_gp"], G0) * gap / BETA) if TEAMS_ADOPTED else bonus0
upd_def = {"playerId": P26["playerId"].to_numpy(), "z": Z_DEF, "bonus": BONUS26.to_dict()}
upd_war = {"playerId": P26["playerId"].to_numpy(), "z": Z_WAR, "bonus": BONUS26.to_dict()}
UPD = SE.run_scenario(B, start_date=TODAY, sit_out=SIT, n_sims=10000, seed=SEED, stat_lines=True, updated=upd_def)
UWAR = SE.run_scenario(B, start_date=TODAY, sit_out=SIT, n_sims=10000, seed=SEED, stat_lines=True, updated=upd_war)
tb = pd.DataFrame({"preseason bonus": bonus0, "updated bonus": BONUS26, "real gp": real26["real_gp"], "gap per game": gap,
                   "opening night": BASE["table"].set_index("team")["points"], "today, not updated": real26["points"],
                   "today, updated": UPD["table"].set_index("team")["points"],
                   "today, + skater WAR": UWAR["table"].set_index("team")["points"],
                   "playoff % not updated": real26["playoff_pct"], "playoff % updated": UPD["table"].set_index("team")["playoff_pct"]})
print(f"Team bonus update {'(adopted)' if TEAMS_ADOPTED else '(not adopted: bonuses unchanged)'}, g0 = {G0:.0f}:")
print(tb.sort_values("today, updated", ascending=False).round(2).to_string())
d0, d1 = real26.loc["DET"], UPD["table"].set_index("team").loc["DET"]
print(f"\nDetroit: opening night {BASE['table'].set_index('team').loc['DET', 'points']:.1f} points ({BASE['table'].set_index('team').loc['DET', 'playoff_pct']:.1f}% playoffs); "
      f"today not updated {d0['points']:.1f} ({d0['playoff_pct']:.1f}%); today updated {d1['points']:.1f} ({d1['playoff_pct']:.1f}%)")

Team bonus update (adopted), g0 = 49:
      preseason bonus  updated bonus  real gp  gap per game  opening night  today, not updated  today, updated  today, + skater WAR  playoff % not updated  playoff % updated
team                                                                                                                                                                         
CAR              0.20           0.39       19          0.25         102.75              109.21          119.45               122.13                  89.55              99.01
EDM             -0.13           0.05       19          0.24          99.49              104.69          114.59               114.75                  90.48              98.45
COL              0.32           0.20       18         -0.17         110.06              108.24          111.08               111.60                  92.76              96.39
MIN              0.19           0.39       19          0.28          98.37              104.

In [15]:
s_pre, _ = SE.season_lines(PRE, B)
s_upd, _ = SE.season_lines(UPD, B)
s_war, _ = SE.season_lines(UWAR, B)
names = ["Connor McDavid", "Leon Draisaitl", "Cutter Gauthier", "Adam Fantilli"] + \
        sorted(lu26.loc[(lu26["team"] == "DET") & lu26["dressed"] & (lu26["group"] != "G"), "name"])
rows = []
for nm in names:
    i = pid(nm)
    p = P26.set_index("playerId").loc[i]
    a, b_, c = s_pre.set_index("playerId").loc[i], s_upd.set_index("playerId").loc[i], s_war.set_index("playerId").loc[i]
    ud, uw = U_DEF.set_index("playerId").loc[i], U_WAR.set_index("playerId").loc[i]
    rows.append({"player": nm, "team": p["team_now"], "GP": int(p["gp"]), "G": int(p["G"]), "A": int(p["A"]),
                 "pace P/84": (p["G"] + p["A"]) / max(p["gp"], 1) * 84,
                 "P/60 pre": p["g60"] + p["a60"], "P/60 upd": ud["g60"] + ud["a60"], "P/60 +WAR": uw["g60"] + uw["a60"],
                 "WAR/60 pre": p["proj_war_60"], "WAR/60 upd": ud["war60"], "WAR/60 +WAR": uw["war60"], "ESS": ud["ess"],
                 "rest P pre": a["sim_p"], "rest P upd": b_["sim_p"], "rest P 10th-90th": f"{b_['sim_p_p10']:.0f}-{b_['sim_p_p90']:.0f}",
                 "season P upd": b_["p"]})
print("Preseason vs. updated (default) vs. with skater WAR evidence; rest of season from today (average of 10,000):")
print(pd.DataFrame(rows).round(3).to_string(index=False))

Preseason vs. updated (default) vs. with skater WAR evidence; rest of season from today (average of 10,000):
                  player team  GP  G  A  pace P/84  P/60 pre  P/60 upd  P/60 +WAR  WAR/60 pre  WAR/60 upd  WAR/60 +WAR      ESS  rest P pre  rest P upd rest P 10th-90th  season P upd
          Connor McDavid  EDM  19 11 37    212.211     4.175     5.387      5.170       0.175       0.240        0.230  637.070      83.272     106.637           84-130       154.637
          Leon Draisaitl  EDM  19 18 18    159.158     3.742     4.401      4.070       0.162       0.204        0.185 1374.092      70.637      82.913           64-104       118.913
         Cutter Gauthier  ANA  17 15  7    108.706     2.743     3.422      2.756       0.119       0.179        0.124 1261.125      51.929      62.542            45-81        84.542
           Adam Fantilli  CBJ  15  4  2     33.600     2.429     2.024      1.778       0.078       0.034        0.001 1608.897      47.788      39.148        

## 10. The forecast log

**Saved:** `data/forecast_log/updated_projections_<date>.csv` (gitignored), one row per 2026-27 pool player.

- **Preseason:** WAR/60, goals/60, assists/60.
- **Updated, default version:** the same three, plus the ESS.
- **Updated, with skater WAR evidence:** the same three.
- **The evidence used:** games, goals, assists and minutes so far; skater WAR/60 so far; goalie WAR/60 so far.
- **Team bonuses:** preseason and updated.

**What it's for:** later notebooks can score these forecasts against what really happens.

In [16]:
LOG_DIR.mkdir(parents=True, exist_ok=True)
log = P26[["playerId", "name", "team", "team_now", "group", "gp", "G", "A", "toi", "war60_so_far", "gwar60_so_far", "proj_war_60", "g60", "a60"]].copy()
log = log.rename(columns={"proj_war_60": "pre_war60", "g60": "pre_g60", "a60": "pre_a60", "toi": "minutes_so_far"})
for tag, U in [("upd", U_DEF), ("upd_war", U_WAR)]:
    u = U.set_index("playerId")
    log[f"{tag}_war60"], log[f"{tag}_g60"], log[f"{tag}_a60"], log[f"{tag}_ess"] = (
        log["playerId"].map(u["war60"]), log["playerId"].map(u["g60"]), log["playerId"].map(u["a60"]), log["playerId"].map(u["ess"]))
log["team_bonus_pre"] = log["team_now"].map(bonus0)
log["team_bonus_upd"] = log["team_now"].map(BONUS26)
log["evidence_through"] = rg["date"].max()
log["version"] = VERSION
log["adopted"] = json.dumps(ADOPT)
path = LOG_DIR / f"updated_projections_{TODAY}.csv"
log.to_csv(path, index=False)
print(f"Saved {len(log):,} players to {path}")
same = SE.run_scenario(B, start_date=TODAY, sit_out=SIT, n_sims=500, seed=3, updated=upd_def)
again = SE.run_scenario(B, start_date=TODAY, sit_out=SIT, n_sims=500, seed=3, updated=upd_def)
assert same["table"].equals(again["table"])
print("Same seed, same results (with the update): yes")

Saved 1,348 players to ../data/forecast_log/updated_projections_2026-11-10.csv
Same seed, same results (with the update): yes


## Takeaways
- **What's adopted for 2026-27:** the skater update (goals and assists so far) and the team bonus update.
  - **Not adopted:** the goalie update.
  - **Never the default:** skater WAR evidence. It's shown as a second version, and both are logged.
- **Whole vs. split** (your change 1): it's essentially a tie, and **training and test disagree**.
  - **Training seasons pick whole:** average points error over the three cutoffs 6.176 vs. 6.181 for split.
  - **Test seasons favor split:** 6.695 vs. 6.713 for whole.
  - **The ranges:** whole's 80% ranges cover 81-83% of real rest-of-season points; split's run wider, 83-87%, overshooting most after 40 games.
  - **Said plainly:** by the rule (pick on training), the default is **whole**, and **Ryan chose to keep it**. The difference is a few hundredths of a point, so the backtest doesn't confirm that split is better.
- **Skaters, test seasons** (rest-of-season points error, whole):

  | | Update | Preseason | Pace |
  |---|---|---|---|
  | After 10 games | 7.93 | 7.99 | 18.55 |
  | After 20 games | 6.96 | 7.18 | 11.85 |
  | After 40 games | 5.25 | 5.54 | 6.76 |

  - It beats both at every cutoff, and the training seasons agree.
  - **Pace adds nothing:** the blend weight on the update is 1.0 at every cutoff.
- **Small samples move little, big ones more:**
  - **Gauthier:** after his first 6 games (2 G, 3 A) he doesn't move; after 17 games (15 G, 7 A), points/60 rises 24%.
  - **McDavid:** +19% after 6 games (17 points), +30% after 19 (48 points).
  - **Fantilli:** -11% after 6 games, -18% after 15.
  - **Said plainly:** your "12 goals in 6 games" isn't Gauthier's real start. The real numbers are above.
- **Goalies** (2021-22 to 2025-26 only; 2013-14 to 2018-19 have no goalie test):
  - **Error:** the update leaves rest-of-season goalie error almost unchanged.
  - **Ranges:** split's 80% ranges cover only 77-79%.
  - **Teams:** adding it makes team predictions **worse** at every cutoff (after 10 games 12.20 vs. 11.85 points; after 40 games 7.21 vs. 7.03). So it isn't adopted.
  - **The data checks out:** goals saved above expected from the shot file matches the goalie WAR table (correlation 0.997).
- **The team bonus update:** g0 = 49 games.
  - It comes from game-to-game luck in a team's points (variance 0.845 per game) and real team effects left after the preseason model (0.0172).
  - So a team's leftover results count 17% after 10 games, 29% after 20, 45% after 40.
  - **Test seasons** (rest-of-season team points error):

    | | Update | Preseason | Pace |
    |---|---|---|---|
    | After 10 games | 11.60 | 12.15 | 24.06 |
    | After 20 games | 10.43 | 11.16 | 15.01 |
    | After 40 games | 7.37 | 8.59 | 8.17 |

    The training seasons agree. Most of the gain comes from the team bonus; the player update alone helps a little.
- **2026-27 today** (evidence through Nov 9):
  - **Points/60, preseason to updated:** McDavid 4.18 to 5.39, Draisaitl 3.74 to 4.40, Gauthier 2.74 to 3.42, Fantilli 2.43 to 2.02. Detroit's skaters move a few percent (Raymond 3.10 to 2.92, DeBrincat 2.87 to 2.79).
  - **With skater WAR evidence** (not the default), some moves shrink: Gauthier 2.76, McDavid 5.17.
  - **Team bonuses move most for the extremes:** Vancouver -0.42 to -0.83, Chicago -0.22 to -0.47, Carolina and Edmonton up about 0.2.
  - **Detroit, from today, with Larkin and Edvinsson sitting:** 80.8 points and a 10.2% playoff chance, updated. Not updated: 84.4 and 17.2%. Opening night: 91.1 and 40.1%.
- **The data:**
  - **6,961 past boxscores,** every game within the first 40 of either team.
  - **28 team-games of about 13,900 are off by one goal:** in the shot-data seasons the boxscore almost always has one more (a goal missing from the shot data). Only boxscores are used as evidence.
- **Checks:**
  - no evidence gives posterior = prior;
  - with a full season left, the split version keeps the prior's spread (0.97-1.02);
  - the same seed gives the same results;
  - notebooks 26 and 27 reproduce exactly with the new code.
- **The forecast log** (`data/forecast_log/updated_projections_2026-11-10.csv`, gitignored): 1,348 players, both versions and the team bonuses. Later notebooks can score it.